#### Data Prep-01: PIE_annot_attrb_val.csv

In [1]:
# library imports
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from tqdm import tqdm

In [2]:
# Configuration
# Path to the PIE dataset annotations_attributes folder
base_dir = r"C:\Users\dthq657\OneDrive - University of Leeds\Research\Datasets\PIE\annotations_attributes"

# Helper function: Parse one XML file
def parse_attributes_xml(xml_path):
    """
    Parse a single XML file containing <pedestrian ...> entries.
    Returns a list of dictionaries, one per pedestrian.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    records = []
    for ped in root.findall(".//pedestrian"):
        ped_data = ped.attrib.copy()
        ped_data["source_file"] = os.path.basename(xml_path)  # track which video it came from
        records.append(ped_data)
    return records

# Find all attribute XML files across sets
xml_files = []
for set_name in sorted(os.listdir(base_dir)):
    set_path = os.path.join(base_dir, set_name)
    if not os.path.isdir(set_path):
        continue
    for file in os.listdir(set_path):
        if file.endswith(".xml"):
            xml_files.append(os.path.join(set_path, file))
print(f"Found {len(xml_files)} attribute XML files across all sets.")

# Parse all XML files and collect records
all_records = []
for xml_file in tqdm(xml_files, desc="Extracting pedestrian attributes"):
    try:
        records = parse_attributes_xml(xml_file)
        all_records.extend(records)
    except Exception as e:
        print(f"⚠️ Error reading {xml_file}: {e}")

# Convert to DataFrame and clean
df_attributes = pd.DataFrame(all_records)
# convert numeric fields to numbers (if they exist)
for col in ["critical_point", "crossing", "crossing_point", "exp_start_point", "intention_prob", "num_lanes"]:
    if col in df_attributes.columns:
        df_attributes[col] = pd.to_numeric(df_attributes[col], errors="coerce")

print(f"✅ Parsed {len(df_attributes)} total pedestrian attribute entries.")
print("Preview Sample:")
df_attributes.head()

Found 53 attribute XML files across all sets.


Extracting pedestrian attributes: 100%|██████████| 53/53 [00:00<00:00, 73.59it/s]

✅ Parsed 1842 total pedestrian attribute entries.
Preview Sample:


,age,critical_point,crossing,crossing_point,exp_start_point,gender,id,intention_prob,intersection,num_lanes,signalized,traffic_direction,source_file
0,adult,1595,0,1613,1568,male,1_1_7,0.866667,T-right,4,n/a,TW,video_0001_attributes.xml
1,adult,15268,0,15298,15180,male,1_1_9,0.916667,four-way,4,CS,TW,video_0001_attributes.xml
2,adult,16543,0,16561,16453,male,1_1_11,0.950000,four-way,4,CS,TW,video_0001_attributes.xml
3,child,1470,0,1516,1406,female,1_1_6,0.716667,T-right,4,n/a,TW,video_0001_attributes.xml
4,adult,17317,-1,17342,17227,male,1_1_12,0.083333,midblock,4,n/a,TW,video_0001_attributes.xml


In [3]:
df_attributes.columns

Index(['age', 'critical_point', 'crossing', 'crossing_point',
       'exp_start_point', 'gender', 'id', 'intention_prob', 'intersection',
       'num_lanes', 'signalized', 'traffic_direction', 'source_file'],
      dtype='object')

In [4]:
val_attributes = df_attributes[df_attributes["id"].astype(str).str.startswith("3_")].copy()
val_attributes.reset_index(drop=True, inplace=True)
val_attributes.head(10)

,age,critical_point,crossing,crossing_point,exp_start_point,gender,id,intention_prob,intersection,num_lanes,signalized,traffic_direction,source_file
0,adult,3434,0,3452,3361,male,3_1_232,0.950000,four-way,3,CS,OW,video_0001_attributes.xml
1,adult,7238,0,7275,7148,male,3_1_237,0.950000,four-way,4,CS,OW,video_0001_attributes.xml
2,adult,9175,0,9180,9141,female,3_1_255,0.916667,four-way,3,CS,OW,video_0001_attributes.xml
3,adult,17023,1,17029,16933,male,3_1_267,0.833333,midblock,3,n/a,OW,video_0001_attributes.xml
4,adult,3260,0,3278,3170,male,3_1_231,0.766667,midblock,3,n/a,OW,video_0001_attributes.xml
5,senior,2808,0,2827,2718,female,3_1_224,0.850000,four-way,4,CS,TW,video_0001_attributes.xml
6,senior,9184,0,9190,9159,female,3_1_262,0.950000,four-way,3,CS,OW,video_0001_attributes.xml
7,adult,17082,1,17041,17008,male,3_1_268,0.900000,midblock,3,n/a,OW,video_0001_attributes.xml
8,adult,2825,0,2833,2782,female,3_1_226,0.966667,four-way,4,CS,TW,video_0001_attributes.xml
9,adult,1833,0,1862,1743,male,3_1_221,0.983333,four-way,4,CS,TW,video_0001_attributes.xml


In [5]:
# Save first required dataset
val_attributes.to_csv("./PIE_annot_attrb_val.csv", index=False)

#### Additional Info:

In [6]:
fp_pedestrians = [
    "3_1_248", "3_1_266", "3_2_294", "3_2_295", "3_2_296", "3_3_315", "3_3_325",
    "3_4_348", "3_4_358", "3_4_362", "3_4_363", "3_5_399", "3_5_402", "3_5_403",
    "3_5_408", "3_5_422", "3_6_462", "3_6_477", "3_7_506", "3_7_510", "3_8_531",
    "3_8_545", "3_8_551", "3_9_585", "3_9_586", "3_9_588", "3_9_611", "3_9_615",
    "3_9_625", "3_10_648"
]

fn_pedestrians = [
    "3_2_290", "3_2_302", "3_2_303", "3_3_309", "3_3_326", "3_3_327", "3_3_337",
    "3_3_341", "3_4_351", "3_4_356", "3_5_396", "3_5_397", "3_5_398", "3_5_419",
    "3_5_420", "3_6_433", "3_6_441", "3_7_509", "3_8_528", "3_8_529", "3_8_530",
    "3_8_532", "3_8_534", "3_8_550", "3_8_552", "3_8_554", "3_8_555", "3_9_587",
    "3_9_616", "3_9_617"
]

In [7]:
# Extract FP set
val_fp_attr = val_attributes[val_attributes["id"].isin(fp_pedestrians)].copy()
val_fp_attr.reset_index(drop=True, inplace=True)
val_fp_attr.head()

,age,critical_point,crossing,crossing_point,exp_start_point,gender,id,intention_prob,intersection,num_lanes,signalized,traffic_direction,source_file
0,adult,16925,-1,17094,16862,male,3_1_266,0.116667,midblock,3,n/a,OW,video_0001_attributes.xml
1,adult,9076,0,9086,8986,male,3_1_248,0.383333,midblock,3,n/a,OW,video_0001_attributes.xml
2,adult,10009,0,10036,9919,female,3_2_296,0.916667,four-way,2,CS,OW,video_0002_attributes.xml
3,adult,10010,0,10039,9974,male,3_2_295,0.883333,four-way,2,CS,OW,video_0002_attributes.xml
4,adult,10009,0,10036,9713,male,3_2_294,0.950000,four-way,2,CS,OW,video_0002_attributes.xml


In [8]:
# Extract FN set
val_fn_attr = val_attributes[val_attributes["id"].isin(fn_pedestrians)].copy()
val_fn_attr.reset_index(drop=True, inplace=True)
val_fn_attr.head()

,age,critical_point,crossing,crossing_point,exp_start_point,gender,id,intention_prob,intersection,num_lanes,signalized,traffic_direction,source_file
0,senior,5561,1,6090,5471,male,3_2_290,0.85,four-way,3,CS,OW,video_0002_attributes.xml
1,adult,11824,1,11971,11781,female,3_2_302,0.95,four-way,2,CS,TW,video_0002_attributes.xml
2,adult,11913,1,11943,11823,male,3_2_303,0.85,four-way,2,CS,TW,video_0002_attributes.xml
3,adult,17581,1,17646,17491,female,3_3_327,0.95,T,2,CS,OW,video_0003_attributes.xml
4,adult,17557,1,17636,17467,male,3_3_326,0.95,T,2,CS,OW,video_0003_attributes.xml


In [9]:
# Save FP and FN sets
val_fp_attr.to_csv('./annot-data/PIE_annot_attrb_val_fp.csv', index=False)
val_fn_attr.to_csv('./annot-data/PIE_annot_attrb_val_fn.csv', index=False)

In [10]:
# Exclude FP and FN
excluded_ids = set(fp_pedestrians + fn_pedestrians)
remaining = val_attributes[~val_attributes["id"].isin(excluded_ids)].copy()
remaining["crossing"] = pd.to_numeric(remaining["crossing"], errors="coerce")

In [11]:
# Extract TP set
val_tp_attr = remaining[remaining["crossing"] == 1].copy()
val_tp_attr.reset_index(drop=True, inplace=True)
val_tp_attr.head()

,age,critical_point,crossing,crossing_point,exp_start_point,gender,id,intention_prob,intersection,num_lanes,signalized,traffic_direction,source_file
0,adult,17023,1,17029,16933,male,3_1_267,0.833333,midblock,3,n/a,OW,video_0001_attributes.xml
1,adult,17082,1,17041,17008,male,3_1_268,0.900000,midblock,3,n/a,OW,video_0001_attributes.xml
2,adult,5625,1,5921,5606,male,3_2_289,0.983333,four-way,3,CS,OW,video_0002_attributes.xml
3,adult,12613,1,12618,12552,male,3_2_304,1.000000,four-way,2,CS,TW,video_0002_attributes.xml
4,adult,12693,1,12711,12651,female,3_2_305,1.000000,four-way,2,CS,TW,video_0002_attributes.xml


In [12]:
# Extract TN set
val_tn_attr = remaining[remaining["crossing"].isin([0, -1])].copy()
val_tn_attr.reset_index(drop=True, inplace=True)
val_tn_attr.head()

,age,critical_point,crossing,crossing_point,exp_start_point,gender,id,intention_prob,intersection,num_lanes,signalized,traffic_direction,source_file
0,adult,3434,0,3452,3361,male,3_1_232,0.950000,four-way,3,CS,OW,video_0001_attributes.xml
1,adult,7238,0,7275,7148,male,3_1_237,0.950000,four-way,4,CS,OW,video_0001_attributes.xml
2,adult,9175,0,9180,9141,female,3_1_255,0.916667,four-way,3,CS,OW,video_0001_attributes.xml
3,adult,3260,0,3278,3170,male,3_1_231,0.766667,midblock,3,n/a,OW,video_0001_attributes.xml
4,senior,2808,0,2827,2718,female,3_1_224,0.850000,four-way,4,CS,TW,video_0001_attributes.xml


In [13]:
# Save TP and TN sets
val_tp_attr.to_csv('./annot-data/PIE_annot_attrb_val_tp.csv', index=False)
val_tn_attr.to_csv('./annot-data/PIE_annot_attrb_val_tn.csv', index=False)